In [1]:
!pip install yfinance pandas numpy scipy matplotlib seaborn dtaidistance scikit-learn

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from dtaidistance import dtw
import warnings
warnings.filterwarnings('ignore')

In [4]:
# --- Configuration ---
BENCHMARK = '^IXIC'
RISK_FREE_RATE = 0.045
START_DATE = '2014-01-01'
END_DATE = '2024-01-01'
ROLLING_WINDOW = 52
N_CLUSTERS = 5  # Target number of clusters
MIN_HISTORY = 400 # Minimum weeks of data required per stock

In [7]:
# Retrieving NASDAQ tickers (100)
nasdaq_100_tickers = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'TSLA', 'GOOGL', 'GOOG', 'AVGO', 'ASML',
    'COST', 'NFLX', 'AZN', 'AMD', 'PEP', 'ADBE', 'LIN', 'QCOM', 'INTU', 'AMAT',
    'CSCO', 'TXN', 'AMGN', 'ISRG', 'MU', 'HON', 'BKNG', 'VRTX', 'REGN', 'ADI',
    'PANW', 'LRCX', 'GILD', 'SBUX', 'MDLZ', 'ADP', 'KLAC', 'MELI', 'SNPS', 'CDNS',
    'CRWD', 'ABNB', 'MAR', 'ORLY', 'CTAS', 'MNST', 'FTNT', 'WDAY', 'MRVL', 'PCAR',
    'CPRT', 'ROST', 'PAYX', 'DXCM', 'KDP', 'ODFL', 'FAST', 'IDXX', 'MCHP', 'BIIB',
    'EA', 'GEHC', 'EXC', 'KHC', 'VRSK', 'ON', 'XEL', 'CTSH', 'DLTR', 'CCEP',
    'CSGP', 'ANSS', 'FANG', 'BKR', 'CDW', 'TEAM', 'DDOG', 'SMCI', 'ILMN', 'WBD',
    'ZS', 'ALGN', 'ENPH', 'GFS', 'MDB', 'LCID', 'RIVN', 'SIRI', 'MTCH', 'PARA'
]

# Downloading benchmark data
print('Downloading NASDAQ benchmark data...')
benchmark_raw = yf.download(BENCHMARK, start=START_DATE, end=END_DATE, interval='1wk')
benchmark_prices = benchmark_raw['Close'].squeeze()
benchmark_returns = np.log(benchmark_prices / benchmark_prices.shift(1)).dropna()

print(f'Benchmark data downloaded: {len(benchmark_returns)} weekly observations')
print(f'Date range: {benchmark_returns.index[0].date()} to {benchmark_returns.index[-1].date()}')

[*********************100%***********************]  1 of 1 completed

Benchmark data downloaded: 521 weekly observations
Date range: 2014-01-08 to 2023-12-27


In [11]:
# Downloading all stock price data
print(f'Downloading price data for {len(nasdaq_100_tickers)} stocks...')

raw_data = yf.download(
    nasdaq_100_tickers,
    start=START_DATE,
    end=END_DATE,
    interval='1wk',
    auto_adjust=True,
    progress=True
)

# Extracting closing prices
price_data = raw_data['Close']

# Resample to weekly frequency (Wednesday) to match Vitulano's methodology 
price_data = price_data.resample('W-WED').last()

# Drop any rows where all values are NaN (holidays/gaps)
price_data = price_data.dropna(how='all')

# Dropping stocks with insufficient history
valid_stocks = price_data.columns[price_data.count() >= MIN_HISTORY].tolist()
price_data = price_data[valid_stocks]

# Fill any small gaps, then drop any remaining NaNs
price_data = price_data.ffill().dropna(axis=1)

print(f'\nStocks with sufficient history: {len(price_data.columns)}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Total weekly observations per stocks: {len(price_data)}')

# DataFrame Corrections
print(price_data.shape)
print(price_data.head())

# Removing any duplicate dates yfinance may have introduced
price_data = price_data[~price_data.index.duplicated(keep='first')]
price_data = price_data.sort_index()

print(f'Shape after reduction: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')

$ANSS: possibly delisted; no timezone found      ]  57 of 90 completed
$PARA: possibly delisted; no timezone found      ]  78 of 90 completed
[*********************100%***********************]  90 of 90 completed

2 Failed downloads:
['ANSS', 'PARA']: possibly delisted; no timezone found



Stocks with sufficient history: 77
Date range: 2014-01-01 to 2023-12-27
Total weekly observations per stocks: 522
(522, 77)
Ticker           AAPL       ADBE        ADI        ADP       ALGN       AMAT  \
Date                                                                           
2014-01-01  16.719610  58.970001  38.374760  54.069004  61.360001  14.819970   
2014-01-08  16.916218  60.369999  38.467613  53.513462  60.400002  15.041789   
2014-01-15  16.999191  60.849998  38.676544  53.761127  62.669998  15.024736   
2014-01-22  15.681222  59.110001  37.539021  51.558994  57.470001  14.290988   
2014-01-29  15.752120  59.720001  36.904453  50.166786  55.230000  14.316586   

Ticker       AMD       AMGN       AMZN       ASML  ...       SIRI   SMCI  \
Date                                               ...                     
2014-01-01  4.18  81.985916  19.901501  79.165085  ...  30.918938  1.677   
2014-01-08  4.30  82.894302  19.877001  80.759216  ...  29.397017  1.703   
2014-01-15